In [19]:
"""
Docling Document Layout Analysis (DLA) + vizualizácia nálezov
Funguje v Jupyter notebooku aj ako standalone skript.

Nastav premenné v sekcii CONFIG nižšie.
"""

# ══════════════════════════════════════════════════════════
#  CONFIG – uprav podľa seba
# ══════════════════════════════════════════════════════════
PDF_PATH  = "data/100_stran_ISLP.pdf"   # cesta k PDF
PAGES     = "1-25"            # "1-3", "1,3,5" alebo None = všetky
OUT_DIR   = "data"     # výstupný adresár
# ══════════════════════════════════════════════════════════

"""
Docling Document Layout Analysis (DLA) + vizualizácia nálezov
Funguje v Jupyter notebooku aj ako standalone skript.

Nastav premenné v sekcii CONFIG nižšie.
"""


import json
import sys
from pathlib import Path
from collections import defaultdict

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.document import DoclingDocument
from docling_core.types.doc import DocItemLabel

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np
from pdf2image import convert_from_path
from PIL import Image

# ── Farby pre každý typ layoutu ──────────────────────────
LABEL_COLORS = {
    "title":                "#e63946",
    "section_header":       "#f4a261",
    "text":                 "#2a9d8f",
    "paragraph":            "#2a9d8f",
    "caption":              "#457b9d",
    "table":                "#8338ec",
    "figure":               "#06d6a0",
    "list_item":            "#ffb703",
    "page_header":          "#adb5bd",
    "page_footer":          "#adb5bd",
    "footnote":             "#6c757d",
    "formula":              "#e9c46a",
    "code":                 "#264653",
    "picture":              "#06d6a0",
    "checkbox_selected":    "#fb8500",
    "checkbox_unselected":  "#fb8500",
}
DEFAULT_COLOR = "#999999"


def parse_pages(pages_str):
    """'1-3,5' → [1, 2, 3, 5]"""
    if pages_str is None:
        return None
    result = []
    for part in str(pages_str).split(","):
        part = part.strip()
        if "-" in part:
            a, b = part.split("-")
            result.extend(range(int(a), int(b) + 1))
        else:
            result.append(int(part))
    return sorted(set(result))


def run_dla(pdf_path, page_numbers=None):
    opts = PdfPipelineOptions()
    opts.do_ocr = False
    opts.do_table_structure = False   # TSR vypnuté
    opts.images_scale = 2.0

    converter = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opts)}
    )

    print(f"⏳  Spúšťam DLA na {Path(pdf_path).name} …")
    result = converter.convert(str(pdf_path))
    doc = result.document
    print(f"✅  DLA hotová – {len(doc.pages)} strán spracovaných")
    return doc


def collect_items_per_page(doc, page_numbers=None):
    """Zozbiera bbox-y; ak page_numbers nie je None, filtruje len tie stranky."""
    pages = defaultdict(list)
    for item, _ in doc.iterate_items():
        if not hasattr(item, "prov") or not item.prov:
            continue
        for prov in item.prov:
            pg = prov.page_no
            if page_numbers and pg not in page_numbers:
                continue
            bbox  = prov.bbox
            label = getattr(item, "label", DocItemLabel.TEXT)
            label_str = label.value if hasattr(label, "value") else str(label)
            text_preview = ""
            if hasattr(item, "text") and item.text:
                text_preview = item.text[:80].replace("\n", " ")
            pages[pg].append({
                "label": label_str,
                "bbox":  (bbox.l, bbox.t, bbox.r, bbox.b),
                "text":  text_preview,
            })
    return pages


def visualize(pdf_path, doc, items_per_page, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_paths = []

    page_numbers_sorted = sorted(items_per_page.keys())
    print("🖼   Renderujem PDF stránky …")
    pil_pages = convert_from_path(
        str(pdf_path), dpi=150,
        first_page=page_numbers_sorted[0],
        last_page=page_numbers_sorted[-1],
    )
    page_img_map = {
        pg: pil_pages[i]
        for i, pg in enumerate(page_numbers_sorted)
        if i < len(pil_pages)
    }

    for pg_no, img in page_img_map.items():
        items = items_per_page.get(pg_no, [])
        if not items:
            continue

        doc_page = doc.pages.get(pg_no)
        if doc_page and doc_page.size:
            pt_w, pt_h = doc_page.size.width, doc_page.size.height
        else:
            pt_w, pt_h = 612, 792

        img_w, img_h = img.size
        sx, sy = img_w / pt_w, img_h / pt_h

        fig, ax = plt.subplots(figsize=(img_w / 100, img_h / 100), dpi=100)
        ax.imshow(np.array(img))
        ax.set_xlim(0, img_w)
        ax.set_ylim(img_h, 0)
        ax.axis("off")

        seen_labels = set()
        for item in items:
            l, t, r, b = item["bbox"]
            color = LABEL_COLORS.get(item["label"], DEFAULT_COLOR)

            # Docling: origin bottom-left → prevedieme na top-left
            x0 = l * sx
            y0 = (pt_h - t) * sy
            w  = (r - l) * sx
            h  = (t - b) * sy

            ax.add_patch(FancyBboxPatch(
                (x0, y0), w, h,
                boxstyle="round,pad=1",
                linewidth=1.5,
                edgecolor=color,
                facecolor=color + "30",
            ))
            ax.text(x0 + 3, y0 + 3, item["label"],
                    fontsize=5.5, color=color, va="top",
                    fontweight="bold", clip_on=True)
            seen_labels.add(item["label"])

        legend_patches = [
            mpatches.Patch(
                facecolor=LABEL_COLORS.get(lbl, DEFAULT_COLOR) + "60",
                edgecolor=LABEL_COLORS.get(lbl, DEFAULT_COLOR),
                label=lbl, linewidth=1.2,
            )
            for lbl in sorted(seen_labels)
        ]
        ax.legend(handles=legend_patches, loc="lower left",
                  fontsize=6, framealpha=0.85, ncol=2, bbox_to_anchor=(0, 0))
        ax.set_title(
            f"DLA – {Path(pdf_path).name}  |  strana {pg_no}  ({len(items)} elementov)",
            fontsize=9, pad=6,
        )

        out_path = out_dir / f"dla_page_{pg_no:03d}.png"
        plt.savefig(out_path, bbox_inches="tight", dpi=150)
        plt.close(fig)
        print(f"   💾  {out_path}")
        out_paths.append(out_path)

    return out_paths


def print_summary(items_per_page):
    total = sum(len(v) for v in items_per_page.values())
    label_counts = defaultdict(int)
    for items in items_per_page.values():
        for it in items:
            label_counts[it["label"]] += 1

    print("\n" + "═" * 56)
    print("  DLA SÚHRN")
    print("═" * 56)
    print(f"  Stránky  : {sorted(items_per_page.keys())}")
    print(f"  Elementy : {total}")
    print()
    for lbl, cnt in sorted(label_counts.items(), key=lambda x: -x[1]):
        bar = "█" * min(cnt, 36)
        print(f"    {lbl:<25} {cnt:>4}  {bar}")
    print("═" * 56 + "\n")


# ── Hlavná logika (spustí sa vždy – notebook aj skript) ──
pdf_path     = Path(PDF_PATH)
page_numbers = parse_pages(PAGES)
out_dir      = Path(OUT_DIR)

if not pdf_path.exists():
    raise FileNotFoundError(f"PDF nenájdené: {pdf_path}")

doc            = run_dla(pdf_path, page_numbers)
items_per_page = collect_items_per_page(doc, page_numbers)

if not items_per_page:
    print("⚠️  Žiadne nálezy. Skús zapnúť OCR: opts.do_ocr = True")
else:
    print_summary(items_per_page)
    imgs = visualize(pdf_path, doc, items_per_page, out_dir)

    # JSON export
    json_path = out_dir / "dla_results.json"
    json_path.write_text(json.dumps(
        {"pdf": str(pdf_path),
         "pages": {str(pg): items for pg, items in items_per_page.items()}},
        ensure_ascii=False, indent=2
    ))
    print(f"📄  JSON: {json_path}")
    print(f"🎉  Hotovo! Výstupy v: {out_dir}/")

    # Zobraz obrázky priamo v notebooku
    try:
        from IPython.display import display, Image as IPImage
        for p in imgs:
            display(IPImage(filename=str(p)))
    except ImportError:
        pass

⏳  Spúšťam DLA na 100_stran_ISLP.pdf …


ValidationError: 1 validation error for DocumentConverter.convert
page_ranges
  Unexpected keyword argument [type=unexpected_keyword_argument, input_value=[(1, 25)], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/unexpected_keyword_argument